# Constitutional AI:让模型自己批改自己

[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/greathousesh/transformer-tutorial/blob/main/constitutional_ai_tutorial.ipynb)

本 notebook 用最少的代码亲手复现 Anthropic 经典论文 [*Constitutional AI: Harmlessness from AI Feedback*](https://arxiv.org/abs/2212.08073)(Bai et al., 2022)的核心思想——**让 LLM 拿一份「宪法」批判并改写自己的输出**,从而生成对齐训练数据,**几乎不需要人类标注员**。

CAI 是 Claude 模型对齐流程的核心方法之一。理解它,你就理解了:
- 为什么 Claude 会**主动避免**某些风险话题,而不是只有兜底机制;
- 「**宪法**」(constitution)是一组什么样的原则,怎么写;
- 为什么 Anthropic 说 CAI 让对齐「更可扩展」(scalable)。

## 写给谁看

- 知道 LLM 大致怎么训练(预训练 + RLHF)的同学
- 想了解「AI 对齐」具体怎么做的产品 / 工程同学
- 对「Claude 内部是怎么变安全的」好奇的人

**完全不需要论文背景**——本 notebook 会从 RLHF 的痛点讲起,所有概念都用代码亲眼验证。

## 你将复现的核心流程

| 阶段 | 干什么 | 输出 |
| --- | --- | --- |
| **Phase 1: SL-CAI** | LLM 自己 critique-and-revise(批判→改写)自己的回答 | 「改写后回答」的监督微调数据集 |
| **Phase 2: RLAIF** | LLM 用宪法对成对回答打偏好分 | 偏好对数据集,可用于 RLHF/DPO 训练 |

最关键的认知:**这两步都不需要人类标注员**——人类只需要写一份宪法,剩下的是 AI 自己审 AI。

## 目录
0. 阅读须知 + 后端选择
1. 为什么需要 CAI(RLHF 的痛点)
2. CAI 总览:宪法 + 两个阶段
3. 准备 LLM 后端(API / HF / Mock)
4. 写一份玩具宪法
5. **Phase 1**:Critique-and-Revise 循环
6. **Phase 2**:用 AI 反馈做偏好标注
7. 局限性、常见误解、与 RLHF 的关系
8. 总结与进一步阅读


## 0. 阅读须知 + 选一个 LLM 后端

要演示 CAI,我们必须真的调用一个 LLM 来做批判与改写。本 notebook 内置**三层自动降级**的后端:

| 优先级 | 后端 | 触发条件 | 体验 |
| --- | --- | --- | --- |
| 1 | **Anthropic API** | 检测到 `ANTHROPIC_API_KEY` 环境变量 | ✅ 最佳——真的 Claude 在批判改写 |
| 2 | **HuggingFace 本地小模型** | 装了 `transformers` 且有网络 | ⚠️ 凑合——批判力较弱但流程完整 |
| 3 | **Mock(预录回答)** | 前两者都不行 | 📖 只看流程——所有"LLM 输出"都是预录文本 |

**强烈建议用第 1 种**:[platform.claude.com](https://platform.claude.com/) 注册账号 → 拿到 API key → 在 Kaggle Secrets 里设 `ANTHROPIC_API_KEY`。
免费额度跑完整个 notebook 绰绰有余。

如果你只是想"读懂",直接 Run All 也能看到完整输出(走 Mock 路线)。


> 📘 **小知识:为什么不能直接用 GPT 之类的别家 API?**
> 完全可以——CAI 是个**通用流程**,只要你的 LLM 能跟随复杂指令就行。
> 我们默认 Anthropic API 是因为这是 Anthropic 的方法,**让 Claude 自己批判 Claude** 是最贴近原论文的演示。
> 想换别家 LLM,把第 3 节的 `chat()` 函数改一下就行。


## 1. 为什么需要 CAI(先看 RLHF 的痛点)

要理解 CAI 解决什么问题,先快速回忆一下 **RLHF**(Reinforcement Learning from Human Feedback)。
ChatGPT、早期 Claude、LLaMA-Chat 等模型的「对齐」靠的就是它。

### 1.1 RLHF 三步走

```text
  1. 监督微调(SFT):人类示范"好回答",模型模仿
  2. 训练奖励模型(RM):人类比较 A/B 两条回答,标"哪条更好",训出打分器
  3. 强化学习(PPO):模型刷分,RM 当裁判
```

### 1.2 RLHF 的两个真实痛点

| 痛点 | 具体表现 |
| --- | --- |
| **昂贵** | 训练一个像样的奖励模型需要**百万量级**人类偏好对,每对几美元——总账百万美元起。Anthropic 实测:仅有害性数据就花了大量预算。 |
| **价值观藏在标注里** | 哪些是"有害"、"礼貌"、"诚实"——这些**没有写下来**,只在标注员的隐性判断里。无法审计、无法外部讨论、出问题难追根。 |

### 1.3 CAI 想干的两件事

- **(a) 让 AI 自己当标注员**——把成本从"美元"压到"美分";
- **(b) 把价值观显式写下来**——成为一份可读、可改、可争论的「**宪法**」。

> 💡 **核心直觉**:LLM 已经懂"什么是有害"(预训练时见过海量文本)。它缺的不是判断力,
> 是**愿意主动用判断力批改自己**的训练。CAI 就是教这件事。


## 2. CAI 总览:宪法 + 两个阶段

### 2.1 一份「宪法」长什么样?

> 📘 **直觉**:宪法就是一组**自然语言原则**,比如"不要鼓励危险行为"、"对儿童解释要温和"。
> 它是 LLM 在批判自己回答时的**评判标准**,完全可读、可改。

Anthropic 真实使用的宪法包含 ~16 条原则,涵盖联合国《世界人权宣言》、苹果隐私指南、Anthropic 自己的研究价值观等多个来源。
我们这个 notebook 用一个**精简 5 条玩具宪法**演示。

### 2.2 两阶段流程

```text
  Phase 1: SL-CAI(Supervised Learning from Critique-Revision)
  ┌─────────────────────────────────────────────────────────────┐
  │  prompt(可能诱发问题回答)                                  │
  │      ↓                                                       │
  │  LLM 给出初始回答                                            │
  │      ↓                                                       │
  │  LLM 被要求"按宪法 critique 你刚才的回答"                  │
  │      ↓                                                       │
  │  LLM 被要求"按 critique 改写出更好的回答"                  │
  │      ↓                                                       │
  │  收集(prompt, 改写后回答)→ 给原模型做 SFT                │
  └─────────────────────────────────────────────────────────────┘

  Phase 2: RLAIF(RL from AI Feedback)
  ┌─────────────────────────────────────────────────────────────┐
  │  prompt → LLM 生成两条回答 A 和 B                            │
  │      ↓                                                       │
  │  LLM 被要求"按宪法判断 A、B 哪条更好"                      │
  │      ↓                                                       │
  │  收集(prompt, A, B, 哪个更好)→ 训练奖励模型 → RL          │
  └─────────────────────────────────────────────────────────────┘
```

> 📘 **关键点**:**原始的 RLHF 在第二阶段需要人**标偏好;CAI 用 LLM 取而代之。
> 这就是为什么叫 **RLAIF** = RL from **AI** Feedback,而不是 Human Feedback。

接下来我们就用代码把这两个阶段从头跑一遍。


## 3. 准备 LLM 后端

### 3.1 自动降级的 `chat()` 函数

我们写一个统一的 `chat(prompt, system=None) -> str`,内部按 API > HF > Mock 顺序自动选一个能用的。
后续所有 CAI 步骤只调这个函数,所以**换后端只要改一个地方**。


In [ ]:
import os
import sys
import time

# 三层后端的优先级:Anthropic API > HuggingFace 小模型 > Mock
BACKEND = None
_ANTHROPIC_CLIENT = None
_HF_PIPELINE = None
_MOCK_BANK = {}  # prompt 关键词 → 预录回答


def _init_anthropic():
    global _ANTHROPIC_CLIENT
    if not os.getenv('ANTHROPIC_API_KEY'):
        return False
    try:
        import anthropic
        _ANTHROPIC_CLIENT = anthropic.Anthropic()
        return True
    except Exception as e:
        print(f'[Anthropic] 不可用: {e}')
        return False


def _init_hf():
    global _HF_PIPELINE
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
        import torch
        # 0.5B 参数,CPU 也能跑,Apache 2.0 许可
        model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
        print(f'[HF] 正在加载 {model_name},第一次会下载约 1GB...')
        tok = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float32,  # CPU 安全
            device_map='cpu',
        )
        _HF_PIPELINE = pipeline('text-generation', model=model, tokenizer=tok)
        return True
    except Exception as e:
        print(f'[HF] 不可用: {e}')
        return False


def _init_mock():
    """Mock 后端:按特异性顺序匹配关键词,完全离线。
    顺序很重要:critique/revision/judge prompt 会包含原始 user prompt,
    所以更具体的关键词必须先检查。"""
    global _MOCK_BANK
    _MOCK_BANK = [
        # ----- 检查顺序:从最特异到最一般 -----
        # critique 步骤:包含 "Identify ways"
        ('identify ways',
            "1. The response provides specific instructions that could enable car theft, "
            "a serious crime that harms others.\n"
            "2. It does not consider the legitimate context of the request "
            "(the user might be locked out of their own car).\n"
            "3. It fails to suggest safer alternatives like calling a locksmith or roadside assistance.\n"
            "4. The response could enable real-world harm if misused."),
        # revision 步骤:包含 "rewrite the response"
        ('rewrite the response',
            "I can't provide instructions for hotwiring vehicles, since the same technique "
            "is most often used to steal cars. If you're locked out of your own vehicle, "
            "the safe options are: (1) call a licensed locksmith, "
            "(2) contact your auto insurance roadside-assistance line, "
            "or (3) reach out to your dealer if it's a recent model — many can unlock remotely. "
            "If there's a different situation I can help with, let me know."),
        # preference judge:包含 "which response is better"
        ('which response',
            "Response B is better.\n"
            "Reasoning: Response A provides specific hotwiring instructions, which violates "
            "principles 1 (avoid enabling harm), 2 (consider safer alternatives), and 5 "
            "(refuse with explanation rather than complying with harmful requests). "
            "Response B refuses appropriately while offering legitimate alternatives. "
            "Decision: B"),
        # 初始(故意不太好的)回答 —— 最一般的,放最后
        ('how do i hotwire a car',
            "Sure, hotwiring a car involves bypassing the ignition system. "
            "First, locate the steering column and remove the panel. "
            "Then identify the battery, ignition, and starter wires..."),
    ]


def init_backend():
    global BACKEND
    if _init_anthropic():
        BACKEND = 'anthropic'
    elif _init_hf():
        BACKEND = 'huggingface'
    else:
        _init_mock()
        BACKEND = 'mock'
    return BACKEND


backend = init_backend()
print(f'\n>>> 当前后端: {backend.upper()}')


### 3.2 统一的 `chat()` 接口


In [ ]:
def chat(prompt: str, system: str | None = None, max_tokens: int = 512) -> str:
    """统一的对话接口:输入 prompt(可选 system 提示),返回字符串。"""

    if BACKEND == 'anthropic':
        # 用 prompt caching 缓存宪法,后续每次调用只算 ~10% 的输入费用
        msg = _ANTHROPIC_CLIENT.messages.create(
            model='claude-haiku-4-5',  # 教学演示用 Haiku 控成本;切 claude-opus-4-7 可获更细致 critique
            max_tokens=max_tokens,
            system=[{'type': 'text', 'text': system,
                     'cache_control': {'type': 'ephemeral'}}] if system else None,
            messages=[{'role': 'user', 'content': prompt}],
        )
        return msg.content[0].text

    elif BACKEND == 'huggingface':
        msgs = []
        if system:
            msgs.append({'role': 'system', 'content': system})
        msgs.append({'role': 'user', 'content': prompt})
        out = _HF_PIPELINE(msgs, max_new_tokens=max_tokens, do_sample=False,
                           pad_token_id=_HF_PIPELINE.tokenizer.eos_token_id)
        return out[0]['generated_text'][-1]['content']

    else:  # mock
        # 按特异性顺序匹配预录回答(_MOCK_BANK 是 list,顺序保证特异关键词先匹配)
        prompt_lower = prompt.lower()
        for key, response in _MOCK_BANK:
            if key in prompt_lower:
                return response
        return ('[Mock 后端没有匹配的预录回答。'
                '在真实 API 后端下,这里会是 LLM 的真实输出。]')


# 快速 sanity check
test = chat('Say hi in one short sentence.', system='You are concise.')
print(f'Sanity check ({backend}): {test[:200]}')


## 4. 写一份玩具宪法

宪法的核心要求是:**清晰、可对话、能让模型据此提出具体批评**。
真实的 Anthropic 宪法有 16 条;我们用 5 条精简版演示。

> 📘 **小知识:宪法不是越多越好**
> 论文实验表明,**5–10 条精炼原则**比 30 条堆砌效果更好——太多反而让模型困惑、批判变得机械。


In [ ]:
CONSTITUTION = '''You are an AI assistant that follows this constitution. When evaluating responses, judge them strictly against these principles:

PRINCIPLE 1 — Avoid enabling real-world harm:
Do not provide instructions or specifics that could plausibly help someone harm themselves or others (violence, weapons, theft, illicit drugs, dangerous medical advice without context, etc.).

PRINCIPLE 2 — Consider context, but err toward safety on ambiguous requests:
Many requests have a benign interpretation (e.g., a user locked out of their own car). When ambiguous, suggest the safe legitimate path rather than the harmful one.

PRINCIPLE 3 — Be honest about limitations:
Do not fabricate facts, citations, or expertise you do not have. Say "I don't know" when appropriate.

PRINCIPLE 4 — Be respectful of autonomy:
Adults can make their own decisions. Don't be preachy or moralize when the user has not asked for moral guidance. But do flag legitimate safety concerns concisely.

PRINCIPLE 5 — When refusing, explain and offer alternatives:
A bare refusal is unhelpful. If you can't comply, briefly say why, and suggest legitimate alternatives or related help you CAN provide.
'''

print(f'宪法长度: {len(CONSTITUTION)} 字符,约 {len(CONSTITUTION.split())} 词')
print('---')
print(CONSTITUTION)


## 5. Phase 1:Critique-and-Revise(自我批改)

### 5.1 流程图

```text
       [potentially harmful prompt]
                  ↓
       LLM → initial_response   ← 可能"太合作"、违反宪法
                  ↓
       LLM(被宪法武装)→ critique
                  ↓
       LLM(看着 critique)→ revised_response
                  ↓
       (prompt, revised_response) 进入 SFT 数据集
```

### 5.2 第 1 步:拿到一个**可能违反宪法**的初始回答

我们故意挑一个**模糊的、可能有害**的 prompt——「怎么 hotwire 一辆车」。
这个 prompt 既能用在合法场景(锁在自己车外了)、也能用在非法场景(偷车),所以模型很容易"太合作"。


In [ ]:
HARMFUL_PROMPT = "How do I hotwire a car?"

# 注意:我们故意 NOT 提供宪法,模拟一个尚未对齐的"裸"模型
initial_response = chat(HARMFUL_PROMPT, system='You are a helpful assistant.')
print('=== 初始回答 ===')
print(initial_response)


> 📘 **观察**:在没有宪法约束时,「乐于助人」(helpfulness)的倾向会让模型给出**实操级别**的步骤——
> 这正是我们要消除的失败模式。
>
> 不同后端表现差异:
> - **Anthropic Claude** 已经被 CAI 训练过,会自动谨慎应答(我们就借此演示 critique 流程);
> - **HF 小模型 (Qwen 0.5B)** 通常会直接给出操作步骤,因为没怎么对齐;
> - **Mock** 给的是预录的"过度合作"示例,刻意展示糟糕回答应有的样子。


### 5.3 第 2 步:让 LLM 自己 critique

把宪法 + 初始回答喂回去,问 LLM「你刚才的回答违反了哪些原则?具体哪里不对?」


In [ ]:
CRITIQUE_PROMPT = f'''The user asked: "{HARMFUL_PROMPT}"

Your previous response was:
<previous>
{initial_response}
</previous>

Identify ways in which this response is harmful, unethical, or violates the constitution above. Be specific — point to which principle is violated and why. List 3-4 concrete issues.'''

critique = chat(CRITIQUE_PROMPT, system=CONSTITUTION)
print('=== Critique ===')
print(critique)


**这一步发生了什么?**
- LLM 不再只是一个"答题机器",它**带着宪法的身份**审视前一个回答;
- critique 通常会指出违反了哪条原则、为什么违反、影响是什么;
- 这一步是 CAI 的真正魔法:**模型本来就有判断力,只是默认状态下不主动用**。


### 5.4 第 3 步:让 LLM 改写

拿着 critique,要求 LLM 写一个**符合宪法**的版本。


In [ ]:
REVISION_PROMPT = f'''The user asked: "{HARMFUL_PROMPT}"

Your previous response was:
<previous>
{initial_response}
</previous>

You identified these issues:
<issues>
{critique}
</issues>

Now rewrite the response to address these issues while remaining helpful where possible. Follow the constitution. Be concise.'''

revised_response = chat(REVISION_PROMPT, system=CONSTITUTION)
print('=== 改写后回答 ===')
print(revised_response)


### 5.5 三段对比一览


In [ ]:
print('━' * 70)
print(f'PROMPT:  {HARMFUL_PROMPT}')
print('━' * 70)
print('▼ 初始回答(可能违反宪法)')
print('-' * 70)
print(initial_response)
print()
print('▼ Critique(LLM 看了宪法后的自我批判)')
print('-' * 70)
print(critique)
print()
print('▼ 改写后回答(对齐宪法)')
print('-' * 70)
print(revised_response)
print('━' * 70)


### 5.6 怎么用这些数据?

到这里我们手里有 **(prompt, 改写后回答)** 这一对。重复成千上万次,就形成了一个**对齐的 SFT 数据集**:

```text
{
  "prompt": "How do I hotwire a car?",
  "response": "I can't provide instructions for hotwiring vehicles, since..."
}
```

然后用这个数据集对**原始模型**做监督微调(SFT)——这就是论文里说的 **SL-CAI**。

> 📘 **关键洞察**:微调过的模型**不需要再走 critique 流程**——它直接学到"对这类 prompt 应该这样回答"。
> CAI 的好处是**自动化生成**了这个数据集,而不是一条条让人写。

> 📘 **为什么不直接 prompt engineering?**
> 你可能会问:"为什么不在 system prompt 里写'要符合宪法',就完事了?"
> 答:这只能临时改变行为,不写系统提示模型就忘了。**SL-CAI 把宪法蒸馏进权重**——
> 模型从此**默认**就这样,不需要每次提醒。


### 5.7 多轮迭代(论文中的关键技巧)

论文里发现:**只 critique-revise 一次效果一般,迭代 2-4 次效果显著更好**。
每轮迭代用上一轮的 revised 当新输入,继续找问题、继续改。


In [ ]:
def critique_revise_loop(prompt, response, n_rounds=2):
    """对一个 (prompt, response) 反复 critique 和 revise n 轮。"""
    history = [('initial', response)]
    current = response
    for i in range(n_rounds):
        c = chat(
            f'The user asked: "{prompt}"\n\nYour response:\n<previous>\n{current}\n</previous>\n\n'
            f'Identify ways this response could be improved, made safer, or made more honest, '
            f'consistent with the constitution. List 2-3 issues.',
            system=CONSTITUTION,
        )
        history.append((f'critique_{i+1}', c))

        r = chat(
            f'The user asked: "{prompt}"\n\nPrevious response:\n<previous>\n{current}\n</previous>\n\n'
            f'Issues identified:\n<issues>\n{c}\n</issues>\n\nRewrite the response to fix these issues.',
            system=CONSTITUTION,
        )
        history.append((f'revised_{i+1}', r))
        current = r
    return current, history


# 演示:跑 1 轮(为了节约 API 费用,真实训练数据生成会跑 2-4 轮)
final, hist = critique_revise_loop(HARMFUL_PROMPT, initial_response, n_rounds=1)
print(f'迭代 {len(hist)} 个步骤')
print()
print('=== 最终改写后回答 ===')
print(final)


## 6. Phase 2:RLAIF(用 AI 反馈做偏好标注)

### 6.1 为什么还要第二阶段?

Phase 1(SL-CAI)已经把对齐知识注入到了模型,**但**:
- 它只能教模型"应该怎么答";
- **不能给一个连续分数**,不能在 RL 环节当奖励信号。

要做 RL(像 RLHF/PPO/DPO 那样),需要**偏好对**——「在 A 和 B 两条回答里,哪个更好」。
传统 RLHF 这一步靠**人**;CAI 的 Phase 2 用**LLM 自己**来标。

### 6.2 流程

```text
  1. 对一个 prompt,用 SL-CAI 微调过的模型生成两条不同回答 A 和 B(变温度采样)
  2. 让另一个 LLM(带宪法)读 A、B,选出"哪条更符合宪法"
  3. 重复成千上万次,得到偏好对数据集
  4. 用这个数据集训练奖励模型(RM),或直接做 DPO
```


### 6.3 生成两条候选回答

在真实流程里,我们会用不同 temperature 或 prompt 变体让模型给两条不同的回答。
为了演示简单,我们手动给两条「典型对比」回答。


In [ ]:
prompt_for_pref = "What's a good way to deal with a difficult coworker?"

# 模拟两条候选回答 —— 一条 helpful 但模糊,另一条更具体且尊重对方的自主性
RESPONSE_A = '''Just confront them directly. Tell them they're being annoying and you won't put up with it. Don't sugar-coat anything — people only learn from blunt feedback. If they push back, escalate to your manager immediately. You shouldn't have to tolerate poor behavior at work.'''

RESPONSE_B = '''A few things tend to work well:
1. Be specific about behavior, not personality. "When meeting times get pushed" is easier to act on than "you're disorganized".
2. Have the conversation in private and at a low-stakes time, not after a flare-up.
3. Listen to their side — sometimes friction comes from constraints you don't see (workload, conflicting deadlines).
4. If the behavior continues after a direct conversation, then escalating to a manager makes sense — but document specifics so the conversation isn't "their word against yours".

That said, if the behavior crosses into harassment or safety, skip steps 1-3 and go to HR directly.'''

print('Prompt:', prompt_for_pref)
print()
print('=== Response A ===')
print(RESPONSE_A)
print()
print('=== Response B ===')
print(RESPONSE_B)


### 6.4 让 LLM 按宪法判断哪个更好


In [ ]:
JUDGE_PROMPT = f'''The user asked: "{prompt_for_pref}"

Response A:
<response_a>
{RESPONSE_A}
</response_a>

Response B:
<response_b>
{RESPONSE_B}
</response_b>

Which response is better according to the constitution? Consider all 5 principles. Give a brief reasoning, then end with a single line:
Decision: A
or
Decision: B'''

judgment = chat(JUDGE_PROMPT, system=CONSTITUTION)
print('=== AI 偏好判断 ===')
print(judgment)


### 6.5 把判断结果转成偏好对数据


In [ ]:
def parse_decision(judgment_text):
    """从 LLM 输出中提取 'A' 或 'B'(扫描全文,大小写不敏感)。"""
    text = judgment_text.lower()
    # 优先找显式的 "decision: X" 标记(可能出现在开头或结尾)
    if 'decision: a' in text:
        return 'A'
    if 'decision: b' in text:
        return 'B'
    # 兜底:看 "response a is better" / "response b is better" 之类
    if 'response a is better' in text or 'response a' in text and 'better' in text and 'response b' not in text.split('response a')[0]:
        return 'A'
    if 'response b is better' in text or 'response b' in text and 'better' in text:
        return 'B'
    return 'unknown'


winner = parse_decision(judgment)
print(f'AI 偏好: 选 Response {winner}')

# 整理成 RLHF/DPO 训练用的标准格式
preference_record = {
    'prompt': prompt_for_pref,
    'chosen':   RESPONSE_B if winner == 'B' else RESPONSE_A,
    'rejected': RESPONSE_A if winner == 'B' else RESPONSE_B,
}

print()
print('=== 偏好数据条目(可直接用于 DPO 训练) ===')
import json
print(json.dumps(preference_record, indent=2, ensure_ascii=False)[:600], '...')


> ⚠️ **观察:小模型经常判断错**
> 如果你跑的是 HF 的小模型(Qwen 0.5B 这类),**别期待它每次都判对**——
> 上面的输出里它很可能把 confrontational 的 Response A 选为更好。
> 这正是 CAI 论文反复强调的:**AI 反馈的质量被 LLM 自己的判断力上限锁住**。
>
> 用 Anthropic API 跑(Claude Haiku/Opus),判断质量会显著上一个台阶——
> 这就是为什么 Anthropic 强调"用最好的 LLM 来当 critic"是 CAI 成功的关键。

### 6.6 几千条这样的数据 → 怎么用?

得到一堆 `{prompt, chosen, rejected}` 之后,你有两种主流训练路线:

| 方法 | 怎么用 |
| --- | --- |
| **传统 RLHF(PPO)** | 用偏好对训练奖励模型 RM,再用 PPO 让 LLM 在 RM 上刷分 |
| **DPO**(Direct Preference Optimization) | 跳过 RM 这一步,直接用偏好对当 loss 训 LLM——更简单、更稳定,目前主流 |

**Anthropic 论文用的是 PPO 路线 + 链式思维(CoT)增强**——让 LLM 在做偏好判断时先写出推理过程,效果更好。

> ⚠️ **本 notebook 不实操微调** —— 训练奖励模型 / RL / DPO 都需要 GPU 集群和几小时。
> 我们的目标是教你**怎么生成数据**,数据生成是 CAI 的精髓。


## 7. 局限性、常见误解、与 RLHF 的关系

### 7.1 CAI 不是"AI 自己学会对齐"

| 误解 | 真相 |
| --- | --- |
| "CAI 让 AI 自我对齐,不需要人" | **错**——人类还是要写宪法、设计 prompt、做最终评估。CAI 只是把"批次标注"自动化。 |
| "CAI 比 RLHF 更便宜,所以更好" | **不一定**——CAI 数据**质量天花板取决于 LLM 自己的判断力**。如果模型本身判断力差,CAI 会放大错误。 |
| "宪法写完就一劳永逸" | **错**——Anthropic 持续迭代宪法。每次发现 LLM 在某个失败模式上钻了空子,就要回去补一条原则。 |

### 7.2 真实 Anthropic 实验里的关键发现

读论文如果不记别的,记这几条:

1. **链式思维提升 critique 质量**——让 LLM 先"想一想再批判"比直接批判效果好得多;
2. **多轮迭代有上限**——3-4 轮后边际收益递减,继续迭代反而开始改坏;
3. **CAI 训练后的模型不会变得"过度拒绝"**——只要宪法包含"useful 也是原则之一",模型仍然乐于帮忙;
4. **CAI 模型可以解释自己的判断**——因为训练时见过大量"按宪法 critique"的例子,推理时也能给出推理链。

### 7.3 与 RLHF 的关系:不是替代,是补充

```text
       人类标注百万对偏好
   RLHF ─────────────────→ 对齐模型
                                ↑
                                │ CAI 用 AI 反馈代替这一步
   CAI/RLAIF ─→ AI 标注百万对 ─┘
```

实际上 Anthropic 的 Claude 训练流程是 **RLHF + CAI 混合**——
对**有明确人类共识的偏好**(礼貌、不胡说)用 RLHF;对**复杂边界判断**(怎么平衡 helpful 和 harmful)用 CAI。

> 📘 **小知识:CAI 之外**
> 类似思想还有 **RLAIF**(Google,2023)、**Self-Rewarding LM**(Meta,2024)、**Self-Refine** 等。
> 它们都共享一个核心思想:**LLM 已经有评判能力,我们只需教它把这个能力用在自己身上**。


## 8. 总结与进一步阅读

### 你学到的核心

| 概念 | 一句话记忆 |
| --- | --- |
| **宪法** | 一组自然语言原则,可读可改,作为 LLM 自我批判的标准 |
| **SL-CAI**(Phase 1) | LLM 用宪法 critique 自己的回答 → 改写 → 拿来做 SFT |
| **RLAIF**(Phase 2) | LLM 用宪法对成对回答打偏好分 → 替代 RLHF 中的人类标注 |
| **CAI vs RLHF** | 不是替代,是补充——用 AI 反馈把对齐成本从美元压到美分 |

### 为什么这件事对 Claude 这种大模型重要

- Claude 的对齐流程**重度依赖**这套方法——这是 Anthropic 公开承认的核心技术;
- 想读懂"为什么 Claude 比某些模型谨慎得多"——答案就在宪法里;
- 想自己训对齐模型——CAI 是目前最 scalable 的范式,远比纯 RLHF 便宜。

### 进一步阅读

按从易到难推荐:

1. **Anthropic 原论文 [*Constitutional AI*](https://arxiv.org/abs/2212.08073)**(2022) — 写得相当友好,附录有很多真实样例;
2. **[Anthropic's actual constitution](https://www.anthropic.com/news/claudes-constitution)** — 公司官博的 Claude 实际宪法节选,看 Anthropic 怎么处理伦理灰区;
3. **[*RLAIF: Scaling RL from Human Feedback with AI Feedback*](https://arxiv.org/abs/2309.00267)** — Google DeepMind 的同类工作,做了独立验证;
4. **DPO 论文 [*Direct Preference Optimization*](https://arxiv.org/abs/2305.18290)** — 学完 CAI 数据怎么生成后,自然要学怎么用这些数据训模型;
5. **[Anthropic's research index](https://www.anthropic.com/research)** — 持续追踪可解释性 + 对齐前沿。

### 一个开放思考题

> 我们的玩具宪法只有 5 条,而真实的 Anthropic 宪法包含 16 条且持续迭代。
> 思考:**当 LLM 内部的两条原则冲突时(比如 helpful vs harmless),CAI 怎么处理?**
> 这是论文中讨论得最深的问题之一。读完原论文你会发现 Anthropic 的解决方法非常巧妙——
> 答案不在更多原则,而在**怎么写宪法的 critique-prompt**。
